# 1. 数据集处理

### 1.1 SCD 数据集

In [ ]:
import wfdb
import numpy as np
from scipy.signal import resample
from torch.utils.data import DataLoader, TensorDataset
import torch
import os

# 全局配置
fs = 250  # 原始采样频率(Hz)
target_fs = 128  # 目标采样频率(Hz)
window_size = 30 * 60 * fs  # 30分钟窗口大小（以样本数表示）
step_size = 1 * 60 * fs  # 1分钟滑动步长（以样本数表示）
dataset_path = os.environ.get("SCD_PATH", os.path.join("data", "SCD"))  # 数据集绝对路径

# 根据表格创建时间配置字典
record_config = {
    '30': {'type': 'SCD',       'start': '6:54:33',     'end': '7:54:33'},
    '31': {'type': 'SCD',       'start': '12:42:24',    'end': '13:42:24'},
    '32': {'type': 'SCD',       'start': '15:45:18',    'end': '16:45:18'},
    '33': {'type': 'SCD',       'start': '3:46:19',     'end': '4:46:19'},
    '34': {'type': 'SCD',       'start': '5:35:44',     'end': '6:35:44'},
    '35': {'type': 'SCD',       'start': '23:34:56',    'end': '24:34:56'},
    '36': {'type': 'SCD',       'start': '17:59:01',    'end': '18:59:01'},
    '37': {'type': 'SCD',       'start': '0:31:13',     'end': '1:31:13'},
    '38': {'type': 'SCD',       'start': '7:01:54',     'end': '8:01:54'},
    '39': {'type': 'SCD',       'start': '3:37:51',     'end': '4:37:51'},
    '40': {'type': 'normal',   'start': 1*3600,        'end': 2*3600},  # 特殊处理1-2小时
    '41': {'type': 'SCD',       'start': '1:59:24',     'end': '2:59:24'},
    '42': {'type': 'normal',    'start': 1*3600,        'end': 2*3600},
    '43': {'type': 'SCD',       'start': '14:37:11',    'end': '15:37:11'},
    '44': {'type': 'SCD',       'start': '18:38:45',    'end': '19:38:45'},
    '45': {'type': 'SCD',       'start': '17:09:17',    'end': '18:09:17'},
    '46': {'type': 'SCD',       'start': '2:41:47',     'end': '3:41:47'},
    '47': {'type': 'SCD',       'start': '5:13:01',     'end': '6:13:01'},
    '48': {'type': 'SCD',       'start': '1:29:40',     'end': '2:29:40'},
    '49': {'type': 'normal',    'start': 1*3600,        'end': 2*3600},
    '50': {'type': 'SCD',       'start': '10:45:43',    'end': '11:45:43'},
    '51': {'type': 'SCD',       'start': '21:58:23',    'end': '22:58:23'},
    '52': {'type': 'SCD',       'start': '1:32:40',     'end': '2:32:40'}
}

In [ ]:
def time_to_seconds(timestr):
    """将时:分:秒或纯秒数转换为总秒数"""
    if isinstance(timestr, int) or isinstance(timestr, float):
        return int(timestr)
    try:
        parts = list(map(int, timestr.split(':')))
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    except:
        raise ValueError(f"无效时间格式: {timestr}")

def extract_class_samples(signal, fs=128, class_minutes=5, segment_seconds=2):
    """
    输入: 5分钟信号  -> 输出: 2秒片段，共150片 (5*60/2)
    signal 已是128Hz
    """
    class_len = class_minutes * 60 * fs              # 5分钟 = 38400点
    segment_len = segment_seconds * fs               # 2秒 = 256点
    num_segments = class_len // segment_len          # =150

    assert len(signal) >= class_len, "🚨信号不足5分钟！"

    signal = signal[:class_len]  # 只取5分钟

    segments = [signal[i*segment_len:(i+1)*segment_len] for i in range(num_segments)]
    return np.array(segments)                        # -> (150,256)



def resample_window(window, original_fs, target_fs):
    """重采样窗口数据"""
    num_samples = int(len(window) * target_fs / original_fs)
    return resample(window, num_samples)

In [ ]:
def check_nan_info(name, array):
    total = np.prod(array.shape)
    nan_count = np.isnan(array).sum()
    inf_count = np.isinf(array).sum()
    zero_count = np.sum(array == 0)

    if nan_count > 0:
        print(f"[N警告] {name} 中存在 NaN: {nan_count}/{total} ({nan_count/total:.4%})")
    if inf_count > 0:
        print(f"[I警告] {name} 中存在 Inf: {inf_count}/{total} ({inf_count/total:.4%})")
    if zero_count == total:
        print(f"[值警告] {name} 全部为0")
    elif zero_count > 0:
        print(f"[值提醒] {name} 中包含 0: {zero_count}/{total} ({zero_count/total:.4%})")


In [ ]:
def fix_nan_with_median(signals):
    """对信号中 NaN/Inf 进行中值填充修复"""
    fixed = signals.copy()
    for ch in range(fixed.shape[1]):
        channel_data = fixed[:, ch]
        finite_mask = np.isfinite(channel_data)
        if not finite_mask.all():
            if np.any(finite_mask):
                median_val = np.median(channel_data[finite_mask])
            else:
                median_val = 0.0  # 如果整列都是NaN/Inf，用0替代
                print(f"⚠️警告：通道{ch} 全为NaN/Inf，使用0填充")
            # 替换 NaN 和 Inf
            channel_data = np.where(np.isfinite(channel_data), channel_data, median_val)
            fixed[:, ch] = channel_data
    return fixed

def fix_nan_1d_with_median(signal_1d):
    """对1D信号中的 NaN 和 Inf 进行中值填充"""
    finite_mask = np.isfinite(signal_1d)
    if not finite_mask.all():
        if np.any(finite_mask):
            median_val = np.median(signal_1d[finite_mask])
        else:
            median_val = 0.0
            print("⚠️ 警告：该信号全为 NaN/Inf，用 0 填充")
        # 替换 NaN 和 Inf
        signal_1d = np.where(np.isfinite(signal_1d), signal_1d, median_val)
    return signal_1d


### 1.2 去噪

In [ ]:
from scipy.signal import butter, filtfilt

# 定义低通滤波函数（4阶Butterworth，截止频率40Hz）
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs  # 奈奎斯特频率
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

# 定义高通滤波函数（4阶Butterworth，截止频率0.5Hz）
def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, data)
    return y

In [ ]:
def process_record(record_id):

    config = record_config[record_id]
    record_path = os.path.join(dataset_path, record_id)

    # 读 ECG 信号
    signals, fields = wfdb.rdsamp(record_path, channels=[1])
    signals = signals[:,0]

    # ===== normal 处理方式 =====
    if config['type'] == 'normal':  
        print(f"处理 Normal 记录 {record_id}")

        signals = fix_nan_1d_with_median(signals)
        signals = resample(signals, int(len(signals) * target_fs / fs))   # 128Hz

        total_5min = 5*60*target_fs   # 38400点

        if len(signals) < total_5min:
            print(f"⚠️ {record_id} 不足5分钟，跳过")
            return None,None

        chunk = signals[:total_5min]               # 直接取前5分钟

        samples = extract_class_samples(chunk, fs=target_fs)   # (150,256)
        labels = np.zeros(len(samples), dtype=int)             # 全0

        return samples, labels



    print(f"处理 SCD 记录 {record_id}")

    end_sec = time_to_seconds(config['end'])
    end_idx = end_sec * fs

    start_idx = end_idx - 35*60*fs             # 前35分钟原逻辑保留
    if start_idx < 0:
        print(f"⚠️ {record_id} 不足35分钟，跳过")
        return None,None

    ecg = signals[start_idx:end_idx]
    ecg = fix_nan_1d_with_median(ecg)
    ecg = resample(ecg, int(len(ecg) * target_fs / fs))         # 128Hz

    class_data=[]
    labels=[]

    # 7类 * 5分钟 = 35分钟，全覆盖
    for c in range(7):
        seg_5min = ecg[c*5*60*target_fs : (c+1)*5*60*target_fs]   # 每类5分钟

        if len(seg_5min) < 5*60*target_fs:
            print(f"⚠️ {record_id} 第{c}类不足5分钟，跳过该类")
            continue

        samples = extract_class_samples(seg_5min, fs=target_fs)   # (150,256)
        class_data.append(samples)
        labels += [c]*len(samples)

    return np.vstack(class_data), np.array(labels)



In [ ]:
all_X=[]
all_Y=[]
for r in record_config:
    X,y = process_record(r)
    if X is not None:
        all_X.append(X)
        all_Y.append(y)

all_X=np.vstack(all_X)
all_Y=np.hstack(all_Y)

print("✔ 处理完成:")
print("数据维度 X =", all_X.shape)
print("标签数量 Y =", all_Y.shape)



In [ ]:
all_X.shape

In [ ]:
import numpy as np

unique, counts = np.unique(all_Y, return_counts=True)
for u,c in zip(unique, counts):
    print(f"类别 {u}: {c} 个样本")


### 1.2 NSR 数据集

In [ ]:
import os
import wfdb
import numpy as np
from scipy.signal import medfilt, butter, filtfilt

NSR_PATH = os.environ.get("NSR_PATH", os.path.join("data", "NSR"))
target_fs = 128


# =============================
# 0) 修复缺省值 (NaN/0→中值替换)
# =============================
def fix_nan_1d_with_median(sig):
    sig = np.array(sig).astype(float)
    sig[np.isnan(sig)] = np.nanmedian(sig)
    zero_idx = np.where(sig == 0)[0]
    if len(zero_idx)>0:
        sig[zero_idx] = np.median(sig)
    return sig


# =============================
# 1) ECG去噪：肌电+工频+基线漂移
# =============================

# 中值滤波去肌电噪声
def denoise_median(sig, kernel=5):
    return medfilt(sig, kernel_size=kernel)

# 0.5–40Hz带通滤波（抗基线漂移/工频干扰）
def bandpass_filter(sig, fs=128, low=0.5, high=40, order=4):
    b,a = butter(order,[low/(fs/2),high/(fs/2)],btype='band')
    return filtfilt(b,a,sig)



# =============================
# 2) 处理单条 NSR 记录
# =============================
def process_nsr_record(record_name):

    record_path = os.path.join(NSR_PATH, record_name)

    try:
        signals, fields = wfdb.rdsamp(record_path, channels=[0])
    except:
        print(f"⚠ 无法读取 {record_name}，跳过")
        return None,None

    ecg = signals[:,0]

    # -------- 去噪处理 -------- #
    ecg = fix_nan_1d_with_median(ecg)
    ecg = denoise_median(ecg, kernel=5)
    ecg = bandpass_filter(ecg, fs=target_fs)

    # --------随机取5分钟信号-------- #
    five_min_len = 5*60*target_fs  # =38400点

    if len(ecg) < five_min_len:
        print(f"⚠ {record_name} 信号不足5分钟，跳过")
        return None,None

    start = np.random.randint(0, len(ecg)-five_min_len)
    chunk = ecg[start:start+five_min_len]

    # --------切为2秒片段 (256点)-------- #
    seg_len = 2 * target_fs      #256
    num_seg = five_min_len // seg_len  #150

    samples = np.array([chunk[i*seg_len:(i+1)*seg_len] for i in range(num_seg)])
    labels = np.zeros(num_seg, dtype=int)

    print(f"✓ NSR {record_name} -> {num_seg} samples (2s each)")
    return samples, labels



# =============================
# 3) 处理整个 NSR 数据集
# =============================
def load_NSR_dataset():
    all_X, all_Y = [], []

    files = sorted([f.replace(".dat","") for f in os.listdir(NSR_PATH) if f.endswith(".dat")])
    print(f"发现 {len(files)} 条 NSR 记录，开始处理...\n")

    for rec in files:
        x,y = process_nsr_record(rec)
        if x is not None:
            all_X.append(x)
            all_Y.append(y)

    all_X = np.vstack(all_X)
    all_Y = np.hstack(all_Y)

    print("\n🎉 NSR全部处理完成")
    print(f"最终样本数: {all_X.shape[0]} (每记录≈150)")
    print(f"信号尺寸:  {all_X.shape} -> (N,256)")
    print(f"标签分布:  {dict(zip(*np.unique(all_Y, return_counts=True)))}\n")

    return all_X, all_Y


In [ ]:
nsr_X, nsr_Y = load_NSR_dataset()

In [ ]:
import numpy as np

# 合并数据
X_total = np.vstack([all_X, nsr_X])   # shape → (SCD样本数 + NSR样本数 , 256)
Y_total = np.hstack([all_Y, nsr_Y])   # shape → 同上

# 保存
np.save("X_total.npy", X_total)
np.save("Y_total.npy", Y_total)

print("✔ 数据已保存到当前目录： X_total.npy   Y_total.npy")
print("X_total:", X_total.shape, "  Y_total:", Y_total.shape)


In [ ]:
import numpy as np

unique, counts = np.unique(Y_total, return_counts=True)
for u,c in zip(unique, counts):
    print(f"类别 {u}: {c} 个样本")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# === 1. 读取数据 ===
X_total = np.load("X_total.npy")   # shape (N, 640)
Y_total = np.load("Y_total.npy")

print("X_total:", X_total.shape, "Y_total:", Y_total.shape)

# === 2. 随机抽取一条样本 ===
idx = random.randint(0, len(X_total)-1)
ecg = X_total[idx]
label = Y_total[idx]

print(f"随机抽样 index={idx}, Label={label}")

# === 3. 时间轴（5秒） ===
fs = 128
t = np.arange(len(ecg)) / fs

# === 4. 画心电图 ===
plt.figure(figsize=(10,3))
plt.plot(t, ecg)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"ECG Sample #{idx}  |  Label={label}")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, medfilt, resample

# ===================== 基础清洗 =====================
def fix_nan(signal):
    finite = np.isfinite(signal)
    if not finite.all():
        med = np.median(signal[finite]) if finite.any() else 0
        signal = np.where(finite, signal, med)
    return signal

# ===================== 去基线漂移（关键！） =====================
def remove_baseline_median(signal, fs=128):
    """
    ECG 专用：双窗口中值滤波去基线
    """
    win1 = int(0.2 * fs) | 1     # 200 ms
    win2 = int(0.6 * fs) | 1     # 600 ms

    baseline = medfilt(signal, win1)
    baseline = medfilt(baseline, win2)

    return signal - baseline

# ===================== 带通滤波 =====================
def bandpass(signal, fs=128, low=0.5, high=40, order=4):
    nyq = fs / 2
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)

# ===================== ECG 预处理主流程 =====================
def preprocess_ecg(signal, orig_fs=128, target_fs=128):
    s = fix_nan(signal)

    # ⭐ 核心：先去基线
    s = remove_baseline_median(s, orig_fs)

    # ⭐ 一步完成平滑 + 去噪
    s = bandpass(s, orig_fs)

    # 可选：重采样
    if orig_fs != target_fs:
        new_len = int(len(s) * target_fs / orig_fs)
        s = resample(s, new_len)

    return s

# ===================== 批量处理 =====================
X_total = np.load("X_total.npy")   # (N, 640)
Y_total = np.load("Y_total.npy")

fs = 128
print(f"加载成功: X_total={X_total.shape}, Y_total={Y_total.shape}")

X_clean = np.zeros_like(X_total)

for i in range(len(X_total)):
    X_clean[i] = preprocess_ecg(X_total[i], orig_fs=fs)

np.save("X_total_clean.npy", X_clean)
print("✨ ECG 去噪完成（基线漂移明显改善）")


In [ ]:
import matplotlib.pyplot as plt

i = 200
plt.figure(figsize=(12,4))
plt.plot(X_total[i], label="Original ECG", alpha=0.6)
plt.plot(X_clean[i], label="Denoised ECG", linewidth=2)
plt.legend()
plt.savefig("Denoised.svg", format='svg', dpi=300)
plt.show()


In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, iirnotch, resample

# ===================== 信号清洗模块 =====================

def fix_nan(signal):
    finite = np.isfinite(signal)
    if not finite.all():
        med = np.median(signal[finite]) if finite.any() else 0
        signal = np.where(finite, signal, med)
    return signal

# ---- 去基线漂移关键 step ----
def highpass(signal, fs=128, cutoff=0.5, order=4):
    nyq = fs/2
    b,a = butter(order, cutoff/nyq, btype='high')
    return filtfilt(b,a,signal)

# ---- 抑制肌电噪声 EMG ----
def lowpass(signal, fs=128, cutoff=40, order=4):
    nyq = fs/2
    b,a = butter(order, cutoff/nyq, btype='low')
    return filtfilt(b,a,signal)

# ---- 工频陷波 50Hz / 60Hz ----
def notch_filter(signal, fs=128, freq=50, Q=30):  
    b,a = iirnotch(freq/(fs/2), Q)  # Q越大越尖锐
    return filtfilt(b,a,signal)

# ===================== 预处理流程调用 =====================
def preprocess_ecg(signal, orig_fs=128, target_fs=128, notch=True):
    s = fix_nan(signal)
    s = highpass(s, orig_fs)       # 去漂移 ⭐重点
    s = lowpass(s, orig_fs)        # 去肌电噪声
    if notch:                      # 工频噪声过滤
        s = notch_filter(s, orig_fs)

    # 若未来混合250Hz数据可重采样
    if orig_fs != target_fs:
        new_len = int(len(s) * target_fs / orig_fs)
        s = resample(s, new_len)

    return s


# ======================================================
# 重新加载并逐条处理，不破坏标签
# ======================================================

X_total = np.load("X_total.npy")       # shape (N, 640)
Y_total = np.load("Y_total.npy")

fs = 128

print(f"加载成功: X_total={X_total.shape}, Y_total={Y_total.shape}")

X_clean = np.zeros_like(X_total)

for i in range(len(X_total)):
    X_clean[i] = preprocess_ecg(X_total[i], orig_fs=fs, target_fs=fs)

np.save("X_total_clean.npy", X_clean)
print("\n✨ 已完成去基线漂移与滤波处理 → 保存为 X_total_clean.npy")


In [ ]:
import matplotlib.pyplot as plt
idx = np.random.randint(0,len(X))

t = np.arange(256)/128

X = np.load("X_total.npy")

plt.figure(figsize=(12,6))
plt.subplot(2,1,1)
plt.plot(t,X[idx])
plt.title("Oraiginal ECG")

plt.subplot(2,1,2)
plt.plot(t,X_clean[idx])
plt.title("Cleaned ECG")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pywt
from scipy.signal import iirnotch, filtfilt, butter


# ================= 工具函数 =================

# 工频陷波
def notch_filter(signal, fs=128, freq=50, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, signal)

# 基线漂移 → 形态学滤波（比高通更稳）
def baseline_remove(signal, fs=128):
    win = int(0.2*fs)              # 200ms窗口
    if win%2==0: win+=1
    from scipy.signal import medfilt
    baseline = medfilt(signal, win)       # 1st stage (large window)
    baseline = medfilt(baseline, int(win/2)*2+1)  # 2nd stage small window
    return signal - baseline

# 肌电噪声 → 小波阈值去噪
def wavelet_denoise(sig, wavelet='db6', level=4):
    coeff = pywt.wavedec(sig, wavelet, level=level)
    sigma = np.median(np.abs(coeff[-1]))/0.6745
    uth = sigma*np.sqrt(2*np.log(len(sig)))
    coeff = [pywt.threshold(c, uth, mode='soft') for c in coeff]
    return pywt.waverec(coeff, wavelet)


# ================= 主处理函数 =================

def process_ecg_batch(X, fs=128):
    cleaned = []
    for ecg in X:
        x = ecg.copy()

        # Step1  notch工频去除
        x = notch_filter(x, fs=fs)

        # Step2  形态学滤波去基线漂移
        x = baseline_remove(x, fs=fs)

        # Step3 小波去肌电噪声（核心提升点🔑）
        x = wavelet_denoise(x)

        cleaned.append(x)

    return np.array(cleaned)



# ================= 处理并保存 =================

X = np.load("X_total.npy", allow_pickle=True)
X_clean = process_ecg_batch(X, fs=128)

np.save("X_total_clean.npy", X_clean)
print("✨ ECG去噪处理完成: X_total_clean.npy 已保存")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

X = np.load("X_total.npy")
Xc = np.load("X_total_clean.npy")

i = np.random.randint(len(X))
plt.figure(figsize=(14,6))

plt.subplot(2,1,1)
plt.title("Original ECG")
plt.plot(X[i])

plt.subplot(2,1,2)
plt.title("Cleaned ECG")
plt.plot(Xc[i])
plt.show()
